![SGSSS Logo](https://github.com/SGSSSonline/text-analysis-for-social-scientists/blob/main/img/SGSSS_Stacked.png?raw=true)

# Practical 1: Preprocessing Text

Computational text analysis allows social scientists to study language at scale — from parliamentary debates and policy documents to social media posts and interview transcripts. But before any analysis can take place, raw text must be transformed into a structured, numerical format that a computer can work with.

This practical introduces the foundational preprocessing steps that underpin nearly all text analysis workflows. We will work through each step using a sample of UK parliamentary speeches (Hansard debates), transforming messy, unstructured text into a clean document-term matrix ready for analysis.

## Aims

This practical has two aims:

1. **Demonstrate how to use Python to preprocess text data** for social science research.
2. **Cultivate computational thinking skills** — breaking down a text preprocessing problem into a series of logical, repeatable steps.

### Lesson Details

| | |
| --- | --- |
| **Level** | Introductory |
| **Time** | ~30 minutes |
| **Pre-requisites** | None |
| **Learning outcomes** | Understand the key steps involved in preprocessing text data |
| | Be able to tokenise text using Python |
| | Be able to apply lowercasing, punctuation removal, and stopword removal |
| | Understand the difference between stemming and lemmatisation |
| | Be able to construct a document-term matrix |
| | Understand TF-IDF weighting and its purpose |

## Guide to Using This Resource

This is a **Jupyter Notebook** — an interactive document that combines text, code, and output in a single environment. If you are viewing this in **Google Colab**, you are running the notebook in the cloud, which means you do not need to install anything on your own machine.

A notebook is made up of **cells**. There are two main types:

- **Markdown cells** contain formatted text (like this one). They provide explanations, instructions, and context.
- **Code cells** contain Python code that you can execute. Code cells are displayed with a grey background and have a play button on the left.

To **run a cell**, click on it and press `Shift+Enter` (or click the play button). The output will appear directly below the cell. You should run the code cells **in order**, from top to bottom, as later cells often depend on variables or modules defined in earlier cells.

If you are using **Google Colab**, make sure to save a copy of this notebook to your own Google Drive first: go to **File > Save a copy in Drive**.

If you are new to Jupyter Notebooks and would like a more detailed introduction, see the excellent materials by Dani Arribas-Bel: [https://github.com/darribas/gds19/blob/master/content/labs/lab_00.ipynb](https://github.com/darribas/gds19/blob/master/content/labs/lab_00.ipynb)

### Interactive Test

Let's make sure everything is working. Run the cell below and enter your name when prompted.

In [ ]:
print("Hello! What is your name?")
name = input()
print(f"Welcome to the course, {name}!")

## Setup

Before we begin, we need to install and import the Python libraries we will use throughout this practical. These include:

- **NLTK** (Natural Language Toolkit) — for tokenisation, stopword removal, stemming, and lemmatisation.
- **pandas** — for working with tabular data.
- **scikit-learn** — for TF-IDF vectorisation.
- **Counter** (from collections) — for counting word frequencies.

In [ ]:
# Install packages (if needed on Colab)
!pip install nltk -q

import nltk
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
import pandas as pd
from collections import Counter

print("Successfully imported necessary modules")

## Loading the Corpus

We will work with a small sample of UK parliamentary speeches drawn from the **Hansard debates** — the official record of proceedings in the UK Parliament. Parliamentary speech data is widely used in social science research for studying political communication, party positioning, policy framing, and more.

The dataset below contains 20 speech excerpts with metadata including the speaker's name, party affiliation, and the date of the speech. These excerpts are illustrative and cover a range of policy topics debated in the House of Commons.

In [ ]:
# Create a sample dataset of parliamentary speech excerpts
speeches_data = {
    "date": [
        "2024-01-15", "2024-01-15", "2024-01-22", "2024-01-22", "2024-02-05",
        "2024-02-05", "2024-02-12", "2024-02-12", "2024-02-19", "2024-02-19",
        "2024-03-04", "2024-03-04", "2024-03-11", "2024-03-11", "2024-03-18",
        "2024-03-18", "2024-03-25", "2024-03-25", "2024-04-01", "2024-04-01"
    ],
    "speaker": [
        "Margaret Thornton", "James Whitfield", "Sarah Okonkwo", "David Hargreaves",
        "Emily Chen", "Robert MacLeod", "Fatima Hussain", "Thomas Brennan",
        "Catherine Lloyd", "Andrew Patel", "Helen Murray", "Kwame Asante",
        "Patricia Gallagher", "Ravi Sharma", "Fiona Blackwood", "Marcus Johnson",
        "Alison Crawford", "Yusuf Ali", "Diana Novak", "Christopher Reeves"
    ],
    "party": [
        "Labour", "Conservative", "Labour", "Conservative", "Labour",
        "SNP", "Labour", "Conservative", "Liberal Democrat", "Labour",
        "SNP", "Labour", "Conservative", "Labour", "SNP",
        "Conservative", "Labour", "Labour", "Liberal Democrat", "Conservative"
    ],
    "speech_text": [
        "The National Health Service remains the cornerstone of our social contract with the British people. We must invest in frontline services and address the chronic staffing shortages that are putting patients at risk. The waiting lists have grown to record levels and this government must take urgent action to reduce them.",
        "Economic growth is the foundation upon which all public services depend. By cutting taxes and reducing unnecessary regulation, we can unleash the potential of British businesses. The private sector, not the state, is the engine of prosperity and job creation in this country.",
        "The climate crisis demands immediate and ambitious action from this House. We cannot afford to delay investment in renewable energy infrastructure. Our children and grandchildren will judge us by the decisions we make today on carbon emissions and environmental protection.",
        "Immigration policy must be fair but firm. We need a points-based system that attracts the skilled workers our economy needs while maintaining control of our borders. The British people voted for sovereignty and we must deliver on that promise.",
        "Education is the great equaliser in our society. Every child, regardless of their background or postcode, deserves access to excellent teaching and modern facilities. We must close the attainment gap between the most and least disadvantaged pupils in our schools.",
        "Scotland's interests are consistently overlooked by this Westminster government. The devolution settlement is being undermined at every turn. The people of Scotland deserve the right to determine their own future and to have their voice heard on matters that affect their daily lives.",
        "The housing crisis is devastating communities across the country. Young people cannot afford to buy their first home and rents are consuming an ever larger share of household incomes. We need a massive programme of social housing construction to address this emergency.",
        "Defence spending must remain a top priority for this government. The threats we face from hostile state actors are growing more complex and more dangerous. We must ensure our armed forces have the equipment and resources they need to keep this nation safe.",
        "Mental health services are chronically underfunded and overstretched. Too many people are waiting months or even years for treatment they desperately need. Parity of esteem between physical and mental health must become a reality, not just a slogan.",
        "The cost of living crisis is hitting working families hardest. Energy bills have skyrocketed and food prices continue to rise at alarming rates. The government must do more to support those who are struggling to heat their homes and feed their children.",
        "The fishing communities of Scotland have been betrayed by broken promises on Brexit. Access to our waters and fair quota allocations are essential for the survival of these coastal communities. We will continue to fight for the rights of Scottish fishermen in this House.",
        "Public transport infrastructure is failing passengers across the north of England. Years of underinvestment have left communities isolated and reliant on expensive and unreliable services. We need a transport revolution that connects people to jobs, education, and opportunity.",
        "Law and order must be restored to our streets. Violent crime and antisocial behaviour are blighting communities and the police need the resources and powers to tackle these problems effectively. We will always stand on the side of victims and law-abiding citizens.",
        "The arts and creative industries contribute billions to our economy and enrich the cultural life of this nation. Cuts to arts funding have devastated regional theatres, museums, and music venues. Investment in culture is not a luxury; it is an investment in our communities and our identity.",
        "Rural communities in Scotland face unique challenges that this government fails to understand. Broadband connectivity, access to healthcare, and affordable transport are not luxuries but necessities. The centralisation of services in urban areas is leaving rural Scotland behind.",
        "Free trade agreements are essential for our post-Brexit economic strategy. British goods and services must have access to global markets on the most favourable terms possible. We are building new trading relationships that will benefit every region of the United Kingdom.",
        "Workers' rights must be strengthened, not eroded, in the modern economy. The rise of zero-hours contracts and the gig economy has created a new class of precarious employment. Every worker deserves fair pay, decent conditions, and the security of knowing their rights are protected by law.",
        "Child poverty is a stain on our national conscience. Over four million children in this country are growing up in poverty, and the numbers are rising. We must reform the welfare system to ensure that no child goes hungry and every family has a decent standard of living.",
        "Local government is the backbone of democracy in this country. Councils are being asked to do more with less, and essential services are being cut to the bone. Proper funding for local authorities is not just desirable; it is essential for the health of our democratic institutions.",
        "Science and innovation are the keys to our future prosperity. The United Kingdom has world-leading universities and research institutions that must be supported and celebrated. Government investment in research and development is an investment in the jobs and industries of tomorrow."
    ]
}

# Create a pandas DataFrame from the dictionary
df = pd.DataFrame(speeches_data)

print(f"Dataset shape: {df.shape[0]} speeches, {df.shape[1]} columns")
df.head()

Let's take a closer look at one of the speeches to understand what we are working with.

In [ ]:
# Select a single speech to use as a working example
sample_text = df["speech_text"].iloc[0]
print(f"Speaker: {df['speaker'].iloc[0]} ({df['party'].iloc[0]})")
print(f"Date: {df['date'].iloc[0]}")
print(f"\nSpeech text:\n{sample_text}")

## Tokenisation

The first step in preprocessing text is **tokenisation** — splitting text into individual units called **tokens**, which are usually words. This is the foundation of all subsequent text analysis: once we have a list of individual tokens, we can count them, filter them, and transform them.

We use NLTK's `word_tokenize()` function, which is more sophisticated than simply splitting on spaces. It handles punctuation, contractions, and other linguistic features intelligently.

In [ ]:
# Tokenise the sample speech
tokens = word_tokenize(sample_text)

# Display the tokens
print("Tokens:")
print(tokens)
print(f"\nNumber of tokens: {len(tokens)}")
print(f"Number of unique tokens: {len(set(tokens))}")

Notice that the tokeniser has split the text into individual words **and** punctuation marks. Punctuation like full stops and commas are treated as separate tokens. This is expected — we will deal with punctuation in the next step.

Also notice the difference between the total number of tokens and the number of unique tokens. This tells us that some words appear more than once in the speech.

## Lowercasing and Punctuation Removal

The next step is to **reduce complexity** in the text. Two common transformations are:

1. **Lowercasing**: Converting all tokens to lowercase so that "The" and "the" are treated as the same word. Unless capitalisation is of analytical interest, this is standard practice.
2. **Punctuation removal**: Removing punctuation marks (commas, full stops, etc.) that are not substantively informative for most text analysis tasks.

We combine these two steps: first we lowercase each token, then we keep only tokens that consist entirely of alphabetic characters (which removes both punctuation and numbers).

In [ ]:
# Convert to lowercase
tokens_lower = [token.lower() for token in tokens]

# Remove punctuation and numbers (keep only alphabetic tokens)
tokens_clean = [token for token in tokens_lower if token.isalpha()]

print("Before cleaning:")
print(f"  Tokens: {len(tokens)}")
print(f"  Unique: {len(set(tokens))}")
print(f"  Sample: {tokens[:10]}")

print("\nAfter lowercasing and punctuation removal:")
print(f"  Tokens: {len(tokens_clean)}")
print(f"  Unique: {len(set(tokens_clean))}")
print(f"  Sample: {tokens_clean[:10]}")

We can see that lowercasing and punctuation removal have reduced both the total number of tokens and the number of unique tokens. This is good — we have removed noise without losing substantive content.

## Stopword Removal

**Stopwords** are common words that appear frequently in any text but carry little substantive meaning — words like "the", "is", "and", "of", "to". These words are essential for grammar but are rarely informative for content analysis.

NLTK provides a built-in list of English stopwords. Removing them helps us focus on the words that actually distinguish one document from another.

In [ ]:
# Load the NLTK English stopword list
stop_words = set(stopwords.words('english'))

# View some of the stopwords
print(f"Number of stopwords in NLTK list: {len(stop_words)}")
print(f"Examples: {sorted(list(stop_words))[:20]}")

In [ ]:
# Remove stopwords from our cleaned tokens
tokens_no_stop = [token for token in tokens_clean if token not in stop_words]

print("Before stopword removal:")
print(f"  Tokens: {len(tokens_clean)}")
print(f"  Sample: {tokens_clean[:10]}")

print("\nAfter stopword removal:")
print(f"  Tokens: {len(tokens_no_stop)}")
print(f"  Sample: {tokens_no_stop[:10]}")

print(f"\nTokens removed: {len(tokens_clean) - len(tokens_no_stop)}")

Stopword removal typically eliminates a large proportion of tokens. The remaining words are more substantively meaningful — they tell us what the speech is actually *about*.

## Stemming vs Lemmatisation

The final preprocessing step is to create **equivalence classes** — grouping together different forms of the same word so they are counted as one. There are two common approaches:

- **Stemming** chops off word endings using simple rules to reduce words to a common root form (called a "stem"). For example, "running", "runs", and "ran" might all be reduced to "run". The result is not always a real word (e.g., "studies" becomes "studi").

- **Lemmatisation** reduces words to their dictionary form (called a "lemma") using knowledge of the language. For example, "better" maps to "good" and "was" maps to "be". The result is always a valid word.

Let's compare both approaches on our sample speech.

In [ ]:
# Initialise the stemmer and lemmatiser
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

# Apply stemming
tokens_stemmed = [stemmer.stem(token) for token in tokens_no_stop]

# Apply lemmatisation
tokens_lemmatised = [lemmatizer.lemmatize(token) for token in tokens_no_stop]

print(f"Original tokens:     {tokens_no_stop[:10]}")
print(f"After stemming:      {tokens_stemmed[:10]}")
print(f"After lemmatisation: {tokens_lemmatised[:10]}")

Let's create a side-by-side comparison table to see the differences more clearly.

In [ ]:
# Create a comparison table showing original, stemmed, and lemmatised forms
comparison = pd.DataFrame({
    "Original": tokens_no_stop,
    "Stemmed": tokens_stemmed,
    "Lemmatised": tokens_lemmatised
})

# Show only rows where stemming or lemmatisation changed the word
changed = comparison[
    (comparison["Original"] != comparison["Stemmed"]) |
    (comparison["Original"] != comparison["Lemmatised"])
]

print("Words that changed after stemming or lemmatisation:")
changed

In [ ]:
# Compare the number of unique terms produced by each approach
print(f"Unique tokens (original):     {len(set(tokens_no_stop))}")
print(f"Unique tokens (stemmed):      {len(set(tokens_stemmed))}")
print(f"Unique tokens (lemmatised):   {len(set(tokens_lemmatised))}")

Stemming tends to be more aggressive, reducing more words to a common form but sometimes producing stems that are not real words (e.g., "servic" for "services"). Lemmatisation is more conservative, producing real dictionary words but potentially missing some groupings. For the rest of this practical, we will use **lemmatisation**, as the results are easier to interpret.

## Document-Term Matrix (DTM)

Now that we know how to preprocess a single document, we can apply these steps to our entire corpus and represent it in a structured numerical format: the **Document-Term Matrix** (DTM).

A DTM is a table where:
- Each **row** represents a document (in our case, a parliamentary speech).
- Each **column** represents a unique term (word) in the corpus.
- Each **cell** contains the count of how many times that term appears in that document.

This is the standard input format for most text analysis methods. If you are familiar with quantitative data, think of it as a dataset where the variables are words and the observations are documents.

First, let's define a function that applies all our preprocessing steps to a single text.

In [ ]:
def preprocess(text):
    """Apply all preprocessing steps to a single text string."""
    # Tokenise
    tokens = word_tokenize(text)
    # Lowercase and remove punctuation/numbers
    tokens = [t.lower() for t in tokens if t.isalpha()]
    # Remove stopwords
    stop_words = set(stopwords.words('english'))
    tokens = [t for t in tokens if t not in stop_words]
    # Lemmatise
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return tokens

Now let's apply this function to every speech in our dataset.

In [ ]:
# Apply preprocessing to all speeches
df["tokens"] = df["speech_text"].apply(preprocess)

# View the first few rows with their tokens
df[["speaker", "party", "tokens"]].head()

Now we can build the DTM using `Counter` to count word frequencies in each document, and then assemble these into a pandas DataFrame.

In [ ]:
# Build the DTM by counting term frequencies in each document
dtm_rows = []
for _, row in df.iterrows():
    word_counts = Counter(row["tokens"])
    dtm_rows.append(word_counts)

# Convert to a DataFrame; fill missing values with 0
dtm = pd.DataFrame(dtm_rows).fillna(0).astype(int)

# Set the index to the speaker names for readability
dtm.index = df["speaker"]

print(f"DTM shape: {dtm.shape[0]} documents x {dtm.shape[1]} terms")
dtm.iloc[:5, :10]  # Show first 5 documents and first 10 terms

The DTM is typically **sparse** — most cells contain zeros because any given word only appears in a small number of documents. This is a natural property of language: authors choose from a large vocabulary, so most words are absent from any single document.

Let's see what the most frequent terms are across the whole corpus.

In [ ]:
# Find the most common terms across all documents
term_totals = dtm.sum(axis=0).sort_values(ascending=False)
print("Top 15 most frequent terms in the corpus:")
term_totals.head(15)

## TF-IDF Weighting

Raw word counts are a useful starting point, but they have a limitation: words that are frequent in one document but also frequent across *all* documents are not very informative. For example, if every parliamentary speech mentions "government", then "government" does not help us distinguish one speech from another.

**TF-IDF** (Term Frequency-Inverse Document Frequency) addresses this by weighting words based on two factors:

- **TF (Term Frequency)**: How often a word appears in a specific document. Higher frequency = higher TF.
- **IDF (Inverse Document Frequency)**: How rare a word is across the entire corpus. Words that appear in fewer documents get a higher IDF score.

The TF-IDF score is the product of these two values: **TF-IDF = TF x IDF**. Words that are frequent in one document but rare across the corpus receive the highest scores — these are the words that best characterise what makes a document distinctive.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Join preprocessed tokens back into strings for the vectoriser
df["clean_text"] = df["tokens"].apply(lambda tokens: " ".join(tokens))

# Create the TF-IDF matrix
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(df["clean_text"])

# Convert to a DataFrame for readability
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=tfidf_vectorizer.get_feature_names_out(),
    index=df["speaker"]
)

print(f"TF-IDF matrix shape: {tfidf_df.shape[0]} documents x {tfidf_df.shape[1]} terms")
tfidf_df.iloc[:5, :10]

Now let's look at the top terms by TF-IDF score for each of the first few speeches. These are the terms that are most distinctive to each speech.

In [ ]:
# Display the top 5 terms by TF-IDF score for each of the first 5 speeches
for i in range(5):
    speaker = df["speaker"].iloc[i]
    party = df["party"].iloc[i]
    top_terms = tfidf_df.iloc[i].sort_values(ascending=False).head(5)
    print(f"{speaker} ({party}):")
    for term, score in top_terms.items():
        print(f"  {term:20s} {score:.4f}")
    print()

The TF-IDF scores highlight the terms that are most *distinctive* to each speech. Compare these with the raw frequency counts from the DTM — TF-IDF downweights common words (like "must" or "government") and upweights terms that are specific to individual speeches (like "health", "tax", or "climate").

## Exercise

Now it's your turn. Your task is to:

1. **Filter the speeches** to select only those by a specific party (e.g., `"SNP"` or `"Conservative"`).
2. **Preprocess** the filtered speeches using the `preprocess()` function we defined earlier.
3. **Create a TF-IDF matrix** for the filtered subset.
4. **Identify the top 10 terms** by average TF-IDF score across the selected speeches.
5. **Reflect**: What do these terms tell you about the content and priorities of that party's speeches?

Use the skeleton code below to guide you. Replace the `# YOUR CODE HERE` comments with your own code.

**Note**: You can also try using your own text data if you prefer — just create a list of text strings and apply the same preprocessing steps.

In [ ]:
# Step 1: Filter speeches by party
party_name = "SNP"  # Change this to another party if you like
# YOUR CODE HERE: create a filtered DataFrame called 'party_df'


In [ ]:
# Step 2: Preprocess the filtered speeches
# YOUR CODE HERE: apply the preprocess() function to the speech_text column


In [ ]:
# Step 3: Create a TF-IDF matrix for the filtered speeches
# YOUR CODE HERE: use TfidfVectorizer on the preprocessed text


In [ ]:
# Step 4: Identify the top 10 terms by average TF-IDF score
# YOUR CODE HERE: calculate the mean TF-IDF score for each term and sort


**Step 5 — Reflection**: What do the top terms tell you about the content and priorities of the party's speeches? Write your answer here.

*YOUR ANSWER HERE*

## Appendix: Exercise Solution

In [ ]:
# --- Exercise Solution ---

# Step 1: Filter speeches by party
party_name = "SNP"
party_df = df[df["party"] == party_name].copy()
print(f"Number of {party_name} speeches: {len(party_df)}")

In [ ]:
# Step 2: Preprocess the filtered speeches
party_df["tokens"] = party_df["speech_text"].apply(preprocess)
party_df["clean_text"] = party_df["tokens"].apply(lambda t: " ".join(t))
party_df[["speaker", "clean_text"]].head()

In [ ]:
# Step 3: Create a TF-IDF matrix for the filtered speeches
party_tfidf_vectorizer = TfidfVectorizer()
party_tfidf_matrix = party_tfidf_vectorizer.fit_transform(party_df["clean_text"])

party_tfidf_df = pd.DataFrame(
    party_tfidf_matrix.toarray(),
    columns=party_tfidf_vectorizer.get_feature_names_out(),
    index=party_df["speaker"]
)

print(f"TF-IDF matrix shape: {party_tfidf_df.shape}")
party_tfidf_df

In [ ]:
# Step 4: Identify the top 10 terms by average TF-IDF score
mean_tfidf = party_tfidf_df.mean(axis=0).sort_values(ascending=False)
print(f"Top 10 terms for {party_name} speeches by average TF-IDF score:")
print(mean_tfidf.head(10))

**Reflection**: The top TF-IDF terms for SNP speeches include words like "scotland", "community", "rural", "fishing", and "right" — reflecting the party's focus on Scottish interests, devolution, and the needs of specific communities. These terms capture the distinctive thematic priorities of the SNP compared to the broader corpus.

## Bibliography

- Grimmer, J., Roberts, M. E., & Stewart, B. M. (2022). *Text as Data: A New Framework for Machine Learning and the Social Sciences*. Princeton University Press.
- Manning, C. D., Raghavan, P., & Schutze, H. (2008). *Introduction to Information Retrieval*. Cambridge University Press.
- Bird, S., Klein, E., & Loper, E. (2009). *Natural Language Processing with Python*. O'Reilly Media.

---

**END OF FILE**